# 1 · Datos del Banco Mundial y capa de KPIs determinista

La capa de KPIs es la **fuente de verdad**: todo número que aparezca en un informe tiene que existir aquí. Datos: World Development Indicators (API v2, sin clave), licencia CC BY 4.0. La instantánea `data/instantanea_wdi.csv` permite reproducir todo sin red.

In [1]:
import sys, os
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
from kpis import capa_kpis, config
import json
print(json.loads(config.METADATOS.read_text(encoding='utf-8'))['descargado'])
df = capa_kpis.cargar(); df.shape

2026-10-27


(6121, 4)

## Cobertura: valores por indicador y año (últimos años)

In [2]:
df[df.anio >= 2019].pivot_table(index='indicador', columns='anio', values='valor', aggfunc='count')

anio,2019,2020,2021,2022,2023,2024,2025
indicador,,,,,,,
crecimiento,18.0,18.0,18.0,18.0,18.0,18.0,18.0
cuenta_corriente,18.0,18.0,18.0,18.0,18.0,18.0,17.0
desempleo,18.0,18.0,18.0,18.0,18.0,18.0,18.0
deuda_externa,13.0,13.0,13.0,13.0,13.0,13.0,NaN
exportaciones,18.0,18.0,18.0,18.0,18.0,18.0,16.0
gini,13.0,12.0,12.0,13.0,14.0,14.0,2.0
ied,18.0,18.0,18.0,18.0,18.0,18.0,18.0
importaciones,18.0,18.0,18.0,18.0,18.0,18.0,16.0
inflacion,17.0,17.0,17.0,17.0,17.0,17.0,16.0


Pobreza, Gini y deuda externa son escasos (no todos los países publican cada año, y la deuda externa de WDI no cubre a los países de ingreso alto). La capa usa el último dato de hasta 3 años antes y lo dice en la tabla.

## Tabla larga: variación interanual, mediana regional, brecha, ranking y anomalía

In [3]:
t = capa_kpis.tabla_larga()
t[(t.iso3 == 'ECU') & (t.anio == 2024)][['indicador', 'valor_previo', 'valor', 'variacion', 'unidad_variacion', 'mediana_region', 'brecha_region', 'ranking', 'n_paises', 'z_anomalia']].round(2)

,indicador,valor_previo,valor,variacion,unidad_variacion,mediana_region,brecha_region,ranking,n_paises,z_anomalia
206,crecimiento,1.830000e+00,-1.940000e+00,-3.78,pp,3.370000e+00,-5.32,18.0,18,-0.62
674,cuenta_corriente,2.040000e+00,5.730000e+00,3.70,pp,-8.900000e-01,6.63,1.0,18,1.95
1141,desempleo,3.510000e+00,3.450000e+00,-0.06,pp,5.320000e+00,-1.86,14.0,18,0.22
1552,deuda_externa,5.211000e+01,4.975000e+01,-2.36,pp,4.182000e+01,7.93,4.0,13,-0.05
1933,exportaciones,2.865000e+01,3.050000e+01,1.85,pp,2.969000e+01,0.81,9.0,18,0.27
2371,gini,4.460000e+01,4.520000e+01,0.60,pp,4.360000e+01,1.60,6.0,14,0.71
2748,ied,4.000000e-01,3.700000e-01,-0.03,pp,2.430000e+00,-2.06,16.0,18,0.24
3215,importaciones,2.851000e+01,2.695000e+01,-1.57,pp,2.962000e+01,-2.67,11.0,18,-0.85
3663,inflacion,2.220000e+00,1.550000e+00,-0.67,pp,4.300000e+00,-2.75,14.0,17,-0.01
4113,pib_pc,6.718140e+03,6.826530e+03,1.61,%,8.238700e+03,-17.14,11.0,18,-1.14


Las variaciones de indicadores en % se expresan en **puntos porcentuales**; las de niveles (PIB en USD, población) en **% relativo**. El ranking es 1 = valor más alto.

## Ranking regional de crecimiento 2024

In [4]:
capa_kpis.ranking_regional('crecimiento', 2024).round(2)

,iso3,valor,ranking,n_paises,pais
0,VEN,5.52,1.0,18,Venezuela
1,DOM,4.95,2.0,18,República Dominicana
2,PRY,4.67,3.0,18,Paraguay
3,CRI,4.08,4.0,18,Costa Rica
4,GTM,3.72,5.0,18,Guatemala
5,HND,3.63,6.0,18,Honduras
6,NIC,3.58,7.0,18,Nicaragua
7,PER,3.52,8.0,18,Perú
8,BRA,3.42,9.0,18,Brasil
9,URY,3.33,10.0,18,Uruguay


## Ecuador frente a la región

In [5]:
c = capa_kpis.comparacion_con_region('ECU', 2024)
c.assign(indicador=c.indicador.map(lambda k: config.IND_POR_CLAVE[k].nombre)).round(2)

,indicador,anio,valor,mediana_region,brecha_region,ranking,n_paises,variacion
0,Crecimiento del PIB real (% anual),2024,-1.940000e+00,3.370000e+00,-5.32,18.0,18,-3.78
1,Saldo de cuenta corriente (% del PIB),2024,5.730000e+00,-8.900000e-01,6.63,1.0,18,3.70
2,"Desempleo (% de la fuerza laboral, estimación ...",2024,3.450000e+00,5.320000e+00,-1.86,14.0,18,-0.06
3,Deuda externa total (% del INB),2024,4.975000e+01,4.182000e+01,7.93,4.0,13,-2.36
4,Exportaciones de bienes y servicios (% del PIB),2024,3.050000e+01,2.969000e+01,0.81,9.0,18,1.85
5,Índice de Gini (0–100),2024,4.520000e+01,4.360000e+01,1.60,6.0,14,0.60
6,"Inversión extranjera directa, entradas netas (...",2024,3.700000e-01,2.430000e+00,-2.06,16.0,18,-0.03
7,Importaciones de bienes y servicios (% del PIB),2024,2.695000e+01,2.962000e+01,-2.67,11.0,18,-1.57
8,"Inflación, precios al consumidor (% anual)",2024,1.550000e+00,4.300000e+00,-2.75,14.0,17,-0.67
9,PIB per cápita (USD corrientes),2024,6.826530e+03,8.238700e+03,-17.14,11.0,18,1.61


![Ecuador vs región](../reports/figures/01_ecuador_vs_region.png)

## Anomalías: cambios fuera de lo normal

z robusto = (variación − mediana histórica) / (1,4826 · MAD), usando **solo años anteriores** (sin mirar el futuro) y un piso de escala (0,5 pp o 2 %) para que series muy suaves, como la población, no marquen décimas como anomalías.

In [6]:
a = capa_kpis.anomalias(desde=2022)
a.round(2)

,indicador,iso3,anio,valor_previo,valor,variacion,unidad_variacion,z_anomalia
0,crecimiento,BOL,2022,1.003000e+01,3.750000e+00,-6.28,pp,-4.81
1,crecimiento,CHL,2022,1.134000e+01,2.060000e+00,-9.27,pp,-5.34
2,crecimiento,HND,2022,1.257000e+01,4.140000e+00,-8.42,pp,-6.21
3,crecimiento,NIC,2022,1.045000e+01,3.550000e+00,-6.90,pp,-4.06
4,crecimiento,PER,2022,1.342000e+01,2.820000e+00,-10.60,pp,-3.99
5,crecimiento,SLV,2022,1.190000e+01,2.950000e+00,-8.95,pp,-9.06
6,desempleo,BOL,2022,5.090000e+00,3.550000e+00,-1.54,pp,-3.05
7,desempleo,BRA,2022,1.316000e+01,9.230000e+00,-3.93,pp,-4.49
8,desempleo,COL,2022,1.390000e+01,1.054000e+01,-3.36,pp,-3.66
9,desempleo,CRI,2022,1.514000e+01,1.132000e+01,-3.81,pp,-3.85


In [7]:
a.groupby('anio').size()

anio
2022    25
2023    20
2024     2
2025     2
dtype: int64

![Anomalías](../reports/figures/02_mapa_anomalias.png)

Los picos de 2009 (crisis financiera) y 2020–2021 (pandemia y rebote) aparecen solos: es una comprobación de que el detector marca lo que un analista esperaría.

## Lo que recibe el LLM

La misma tabla, con las cifras ya formateadas en español.

In [8]:
print(capa_kpis.tabla_para_prompt('ECU', 2024))

| indicador (clave) | año | valor | año anterior | variación | mediana región | brecha vs región | puesto (1 = más alto) | ¿anomalía? |
|---|---|---|---|---|---|---|---|---|
| PIB (USD corrientes) (pib_usd) | 2024 | 123,8 mil millones de USD | 120,8 mil millones de USD | +2,5 % | 116,9 mil millones de USD | +5,9 % | puesto 8 de 18 | no |
| PIB per cápita (USD corrientes) (pib_pc) | 2024 | 6.827 USD | 6.718 USD | +1,6 % | 8.239 USD | -17,1 % | puesto 11 de 18 | no |
| Crecimiento del PIB real (% anual) (crecimiento) | 2024 | -1,9 % | 1,8 % | -3,8 pp | 3,4 % | -5,3 pp | puesto 18 de 18 | no |
| Inflación, precios al consumidor (% anual) (inflacion) | 2024 | 1,5 % | 2,2 % | -0,7 pp | 4,3 % | -2,8 pp | puesto 14 de 17 | no |
| Desempleo (% de la fuerza laboral, estimación OIT) (desempleo) | 2024 | 3,5 % | 3,5 % | -0,1 pp | 5,3 % | -1,9 pp | puesto 14 de 18 | no |
| Exportaciones de bienes y servicios (% del PIB) (exportaciones) | 2024 | 30,5 % | 28,6 % | +1,8 pp | 29,7 % | +0,8 pp | puesto